In [ ]:
## Fine Tuning of a Model distilbert-base-uncased

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch transformers datasets scikit-learn accelerate

In [3]:
import torch
print(torch.__version__)

2.9.0+cpu


In [4]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
df = pd.read_csv("/content/bbc_data.csv")

In [6]:
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nLabel distribution:")
print(df['labels'].value_counts())

Shape: (2225, 2)

Column names: ['data', 'labels']

First few rows:
                                                data         labels
0  Musicians to tackle US red tape  Musicians gro...  entertainment
1  U2s desire to be number one  U2, who have won ...  entertainment
2  Rocker Doherty in on-stage fight  Rock singer ...  entertainment
3  Snicket tops US box office chart  The film ada...  entertainment
4  Oceans Twelve raids box office  Oceans Twelve,...  entertainment

Label distribution:
labels
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [7]:
label_list = df['labels'].unique().tolist()
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print("Label mapping:")
print(label2id)
print()
print("ID mapping:")
print(id2label)

Label mapping:
{'entertainment': 0, 'business': 1, 'sport': 2, 'politics': 3, 'tech': 4}

ID mapping:
{0: 'entertainment', 1: 'business', 2: 'sport', 3: 'politics', 4: 'tech'}


In [8]:
df['label_encoded'] = df['labels'].map(label2id)

# Split into train and test (80-20 split)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['labels'])

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")
print("\nTrain label distribution:")
print(train_df['labels'].value_counts())

Train size: 1780
Test size: 445

Train label distribution:
labels
sport            409
business         408
politics         333
tech             321
entertainment    309
Name: count, dtype: int64


In [9]:
# Load the tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# Load model with 5 output labels (for your 5 categories)
num_labels = len(label2id)
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
print(f"Model loaded: {model_name}")
print(f"Number of labels: {num_labels}")
print(f"Labels: {list(label2id.keys())}")

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
model.to(device)

Model loaded: distilbert-base-uncased
Number of labels: 5
Labels: ['entertainment', 'business', 'sport', 'politics', 'tech']

Using device: cpu


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [11]:
def tokenize_function(texts):
    """Tokenize the input texts"""
    return tokenizer(
        texts,
        padding='max_length',  # Pad all sequences to same length
        truncation=True,       # Cut off text that's too long
        max_length=512,        # Maximum sequence length
        return_tensors='pt'    # Return PyTorch tensors
    )

# Tokenize a sample to see what it looks like
sample_text = train_df['data'].iloc[0]
print("Original text (first 200 chars):")
print(sample_text[:200])
print("\n" + "="*50 + "\n")

# Tokenize it
tokenized = tokenize_function([sample_text])
print("Tokenized output keys:", tokenized.keys())
print("Input IDs shape:", tokenized['input_ids'].shape)
print("First 20 token IDs:", tokenized['input_ids'][0][:20].tolist())

Original text (first 200 chars):
Fiat chief takes steering wheel  The chief executive of the Fiat conglomerate has taken day-to-day control of its struggling car business in an effort to turn it around.  Sergio Marchionne has replace


Tokenized output keys: KeysView({'input_ids': tensor([[  101, 18550,  2708,  3138,  9602,  5217,  1996,  2708,  3237,  1997,
          1996, 18550, 22453,  2038,  2579,  2154,  1011,  2000,  1011,  2154,
          2491,  1997,  2049,  8084,  2482,  2449,  1999,  2019,  3947,  2000,
          2735,  2009,  2105,  1012, 13983,  2233,  3258,  2638,  2038,  2999,
          7253, 17183,  2884,  2004,  2708,  3237,  1997, 18550,  8285,  1010,
          2007,  2720, 17183,  2884,  2975,  1996,  2194,  1012,  2720,  2233,
          3258,  2638,  4150,  1996,  2959,  2132,  1997,  1996,  2449,  1011,
          2029,  2003,  3517,  2000,  2191,  1037,  5385,  2213,  9944,  1006,
          1002, 26314,  2078,  1007,  3279,  1999,  2432,  1011,  1999,  2004,
       

In [12]:
class BBCDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.loc[idx, 'data']
        label = self.data.loc[idx, 'label_encoded']

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = BBCDataset(train_df, tokenizer)
test_dataset = BBCDataset(test_df, tokenizer)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print("\nSample from train dataset:")
print(train_dataset[0])

Train dataset size: 1780
Test dataset size: 445

Sample from train dataset:
{'input_ids': tensor([  101, 18550,  2708,  3138,  9602,  5217,  1996,  2708,  3237,  1997,
         1996, 18550, 22453,  2038,  2579,  2154,  1011,  2000,  1011,  2154,
         2491,  1997,  2049,  8084,  2482,  2449,  1999,  2019,  3947,  2000,
         2735,  2009,  2105,  1012, 13983,  2233,  3258,  2638,  2038,  2999,
         7253, 17183,  2884,  2004,  2708,  3237,  1997, 18550,  8285,  1010,
         2007,  2720, 17183,  2884,  2975,  1996,  2194,  1012,  2720,  2233,
         3258,  2638,  4150,  1996,  2959,  2132,  1997,  1996,  2449,  1011,
         2029,  2003,  3517,  2000,  2191,  1037,  5385,  2213,  9944,  1006,
         1002, 26314,  2078,  1007,  3279,  1999,  2432,  1011,  1999,  2004,
         2116,  2086,  1012, 18550,  2104,  4842, 29021,  1996,  3006,  1999,
         2885,  2197,  2095,  1010,  3773,  4257,  4341,  1012,  1996,  2482,
         2449,  2038,  2081,  2019,  4082,  3279,  1

In [13]:
def compute_metrics(pred):
    """Calculate accuracy, precision, recall, F1"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Test it with a small prediction
print("Evaluation function created successfully!")

Evaluation function created successfully!


In [14]:
# Create a Trainer for evaluation only
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='./results',
        per_device_eval_batch_size=16,
    ),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Evaluate the pre-trained model
print("Evaluating pre-trained model...")
pre_trained_results = trainer.evaluate()
print("\n" + "="*50)
print("PRE-TRAINED MODEL RESULTS (Before Fine-tuning):")
print("="*50)
for key, value in pre_trained_results.items():
    print(f"{key}: {value:.4f}")

Evaluating pre-trained model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



PRE-TRAINED MODEL RESULTS (Before Fine-tuning):
eval_loss: 1.6065
eval_model_preparation_time: 0.0024
eval_accuracy: 0.2270
eval_f1: 0.1584
eval_precision: 0.1867
eval_recall: 0.2270
eval_runtime: 326.9040
eval_samples_per_second: 1.3610
eval_steps_per_second: 0.0860


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
# Let's test on a real example from test set
test_example = test_df.iloc[0]
text = test_example['data']
true_label = test_example['labels']
true_label_id = test_example['label_encoded']

print("Article text (first 300 chars):")
print(text[:300])
print("\n" + "="*50)
print(f"True Category: {true_label}")
print("="*50)

# Tokenize
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
inputs = {key: val.to(device) for key, val in inputs.items()}

# Predict
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=-1).item()
predicted_label = id2label[predicted_class]

# Show results
print("\nPrediction Scores (logits) for each category:")
for i, (label_name, score) in enumerate(zip(label2id.keys(), logits[0].tolist())):
    marker = "👉" if i == predicted_class else "  "
    print(f"{marker} {label_name}: {score:.4f}")

print("\n" + "="*50)
print(f"Model Predicted: {predicted_label}")
print(f"True Category: {true_label}")
print(f"Correct? {'✅ YES' if predicted_label == true_label else '❌ NO'}")
print("="*50)

Article text (first 300 chars):
Rank set to sell off film unit  Leisure group Rank could unveil plans to demerge its film services unit and sell its media business, reports claim.  Rank, formerly famous for the Carry On series, will expose the shake-up at the announcement of its results on Friday, the Sunday Telegraph reported. Ad

True Category: business

Prediction Scores (logits) for each category:
   entertainment: -0.1118
   business: -0.0358
   sport: 0.0000
👉 politics: 0.0759
   tech: 0.0192

Model Predicted: politics
True Category: business
Correct? ❌ NO


In [16]:
# Test 5 random examples
import random
random.seed(42)
test_indices = random.sample(range(len(test_df)), 5)

for idx in test_indices:
    test_example = test_df.iloc[idx]
    text = test_example['data']
    true_label = test_example['labels']

    # Predict
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_class]

    # Show compact results
    correct = "✅" if predicted_label == true_label else "❌"
    print(f"{correct} True: {true_label:15} | Predicted: {predicted_label:15} | Text: {text[:60]}...")

❌ True: business        | Predicted: politics        | Text: Cannabis hopes for drug firm  A prescription cannabis drug m...
❌ True: entertainment   | Predicted: politics        | Text: Singer Christina Aguilera to wed  Pop star Christina Aguiler...
❌ True: sport           | Predicted: politics        | Text: Sculthorpe wants Lions captaincy  Paul Sculthorpe has admitt...
❌ True: sport           | Predicted: politics        | Text: Campbell to extend sprint career  Darren Campbell has set hi...
❌ True: business        | Predicted: tech            | Text: Nortel in $300m profit revision  Telecoms equipment maker No...


In [18]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

print("Training configuration set!")
print(f"Total training samples: {len(train_dataset)}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Number of epochs: {training_args.num_train_epochs}")
print(f"Total training steps: {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training configuration set!
Total training samples: 1780
Batch size: 8
Number of epochs: 3
Total training steps: 666


In [19]:
# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully!")
print("\nStarting fine-tuning...")
print("This will take several minutes. You'llb see progress updates every 50 steps.")
print("="*60)

# Start training!
trainer.train()

Trainer created successfully!

Starting fine-tuning...
This will take several minutes. You'll see progress updates every 50 steps.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.092992,0.139024,0.968539,0.968286,0.968970,0.968539
2,0.099797,0.113742,0.979775,0.979740,0.980291,0.979775
3,0.030689,0.094264,0.982022,0.981939,0.982186,0.982022


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=669, training_loss=0.19400375536315823, metrics={'train_runtime': 13363.6312, 'train_samples_per_second': 0.4, 'train_steps_per_second': 0.05, 'total_flos': 707413753958400.0, 'train_loss': 0.19400375536315823, 'epoch': 3.0})

In [21]:
print("="*60)
print("EVALUATING FINE-TUNED MODEL")
print("="*60)

# Evaluate the fine-tuned model
finetuned_results = trainer.evaluate()

print("\n" + "="*60)
print("FINE-TUNED MODEL RESULTS (After Training):")
print("="*60)
for key, value in finetuned_results.items():
    print(f"{key}: {value:.4f}")

EVALUATING FINE-TUNED MODEL


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



FINE-TUNED MODEL RESULTS (After Training):
eval_loss: 0.0943
eval_accuracy: 0.9820
eval_f1: 0.9819
eval_precision: 0.9822
eval_recall: 0.9820
eval_runtime: 272.7631
eval_samples_per_second: 1.6310
eval_steps_per_second: 0.1030
epoch: 3.0000


In [22]:
print("="*60)
print("TESTING FINE-TUNED MODEL ON REAL EXAMPLES")
print("="*60)

# Test on the same 5 examples we tested before
import random
random.seed(42)
test_indices = random.sample(range(len(test_df)), 5)

correct_count = 0

for idx in test_indices:
    test_example = test_df.iloc[idx]
    text = test_example['data']
    true_label = test_example['labels']

    # Predict
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_class]

    # Check if correct
    is_correct = predicted_label == true_label
    if is_correct:
        correct_count += 1

    correct_marker = "✅" if is_correct else "❌"
    print(f"{correct_marker} True: {true_label:15} | Predicted: {predicted_label:15} | Text: {text[:60]}...")

print("\n" + "="*60)
print(f"Results: {correct_count}/5 correct ({correct_count/5*100:.0f}%)")
print("="*60)

TESTING FINE-TUNED MODEL ON REAL EXAMPLES
✅ True: business        | Predicted: business        | Text: Cannabis hopes for drug firm  A prescription cannabis drug m...
✅ True: entertainment   | Predicted: entertainment   | Text: Singer Christina Aguilera to wed  Pop star Christina Aguiler...
✅ True: sport           | Predicted: sport           | Text: Sculthorpe wants Lions captaincy  Paul Sculthorpe has admitt...
✅ True: sport           | Predicted: sport           | Text: Campbell to extend sprint career  Darren Campbell has set hi...
✅ True: business        | Predicted: business        | Text: Nortel in $300m profit revision  Telecoms equipment maker No...

Results: 5/5 correct (100%)


In [23]:
# Save the fine-tuned model and tokenizer
save_directory = "./finetuned_distilbert_bbc"

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print("="*60)
print("MODEL SAVED SUCCESSFULLY!")
print("="*60)
print(f"Location: {save_directory}")
print("\nSaved files:")
print("  - config.json (model configuration)")
print("  - model.safetensors (model weights)")
print("  - tokenizer files (vocabulary and settings)")
print("="*60)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL SAVED SUCCESSFULLY!
Location: ./finetuned_distilbert_bbc

Saved files:
  - config.json (model configuration)
  - model.safetensors (model weights)
  - tokenizer files (vocabulary and settings)
